# Step 4：隐含波动率与波动率模型

本 Notebook 使用期权收盘价反解 Black-76 IV，选取 OTM 样本，并按每日、每到期日拟合二次波动率或 SVI 模型。

默认二次模型为 $\sigma(k)=a+bk+\tfrac12ck^2$，其中 $k=\ln(K/F)$；IV曲线使用 Forward Delta 定位 25C、75P，并同时标出 ATM $k=0$。

## 当前模块参数值

参数默认值统一在 `00_config.ipynb` 设置；本 cell 只打印当前内核中的实际值。


In [ ]:
_module_parameter_names = ["VOL_MODEL", "MIN_OTM_OPTIONS_PER_EXPIRY", "MIN_OPTION_PRICE", "MIN_IV", "MAX_IV", "SVI_PARAMETER_BOUNDS", "VOL_SURFACE_DELTA_BASIS", "VOL_SURFACE_TARGET_DELTAS", "ATM_LOG_MONEYNESS", "IV_SOLVER_LOWER_BOUND", "IV_SOLVER_TOLERANCE", "IV_SOLVER_MAX_ITERATIONS", "MODEL_SKEW_TAG", "VOLATILITY_MODEL_OUTPUT_PATH", "IV_CURVE_FIGURE_PATH", "SAVE_CSV", "SAVE_FIGURE", "FIGURE_FORMAT"]
print(f'04_volatility_model.ipynb 当前参数：')
for _parameter_name in _module_parameter_names:
    print(f'{_parameter_name} = {globals()[_parameter_name]!r}')


In [ ]:
import math
import warnings
from pathlib import Path
from typing import Callable, Optional

import numpy as np
import pandas as pd

try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    warnings.warn('matplotlib 未安装，将跳过IV曲线图片，但不影响模型结果', RuntimeWarning)

_required_config = {
    'VOL_MODEL', 'MIN_OTM_OPTIONS_PER_EXPIRY', 'MIN_OPTION_PRICE',
    'MIN_IV', 'MAX_IV', 'SVI_PARAMETER_BOUNDS',
    'VOL_SURFACE_DELTA_BASIS', 'VOL_SURFACE_TARGET_DELTAS',
    'ATM_LOG_MONEYNESS', 'IV_SOLVER_LOWER_BOUND',
    'IV_SOLVER_TOLERANCE', 'IV_SOLVER_MAX_ITERATIONS',
    'MODEL_SKEW_TAG', 'VOLATILITY_MODEL_OUTPUT_PATH',
    'IV_CURVE_FIGURE_PATH', 'SAVE_CSV', 'SAVE_FIGURE', 'FIGURE_FORMAT',
}
_required_functions = {'black76_price', 'black76_delta', 'is_otm_option'}
_required_data = {'option_forward_panel'}
_ipython = get_ipython() if 'get_ipython' in globals() else None
if not _required_config.issubset(globals()):
    if _ipython is None:
        raise RuntimeError('请先运行 00_config.ipynb')
    _ipython.run_line_magic('run', './00_config.ipynb')
if not _required_functions.issubset(globals()):
    if _ipython is None:
        raise RuntimeError('请先运行 02_basic_functions.ipynb')
    _ipython.run_line_magic('run', './02_basic_functions.ipynb')
if not _required_data.issubset(globals()):
    if _ipython is None:
        raise RuntimeError('请先运行 03_repo_forward.ipynb')
    _ipython.run_line_magic('run', './03_repo_forward.ipynb')

## Black-76 隐含波动率

先检查无套利价格区间，再用二分法求解 $V_{Black76}(\sigma)=Price$。无法反解的记录保留状态原因，不静默删除。

In [ ]:
# 使用有界二分法从期权收盘价反解 Black-76 隐含波动率。
def black76_implied_volatility(
    price: float, F: float, K: float, tau: float, r: float, option_type: str,
    lower_vol: float = 1e-6, upper_vol: float = 5.0,
    tolerance: float = 1e-8, max_iterations: int = 200,
) -> float:
    """验证 Black-76 无套利边界后，以二分法求解隐含波动率。"""
    values = [price, F, K, tau, r, lower_vol, upper_vol]
    if not all(np.isfinite(values)):
        raise ValueError('IV输入包含非有限值')
    if price <= 0 or F <= 0 or K <= 0 or tau <= 0:
        raise ValueError('PRICE、FORWARD、STRIKE和TAU必须大于零')
    if lower_vol <= 0 or upper_vol <= lower_vol:
        raise ValueError('IV求解上下界非法')
    option_type = _normalize_option_type(option_type)
    discount = math.exp(-r * tau)
    if option_type == 'CALL':
        lower_price = discount * max(F - K, 0.0)
        upper_price = discount * F
    else:
        lower_price = discount * max(K - F, 0.0)
        upper_price = discount * K
    price_tolerance = max(tolerance, 1e-12 * max(1.0, upper_price))
    if price <= lower_price + price_tolerance:
        raise ValueError('价格不高于Black-76内在价值边界')
    if price >= upper_price - price_tolerance:
        raise ValueError('价格不低于Black-76理论上界')
    low_price = black76_price(F, K, lower_vol, tau, r, option_type)
    high_price = black76_price(F, K, upper_vol, tau, r, option_type)
    if price < low_price - price_tolerance or price > high_price + price_tolerance:
        raise ValueError('价格不在配置的IV搜索区间内')
    low, high = lower_vol, upper_vol
    for _ in range(max_iterations):
        mid = 0.5 * (low + high)
        model_price = black76_price(F, K, mid, tau, r, option_type)
        if abs(model_price - price) <= tolerance:
            return float(mid)
        if model_price < price:
            low = mid
        else:
            high = mid
    return float(0.5 * (low + high))

# 为期权Forward面板逐行计算log-moneyness、OTM标记和收盘价隐含波动率。
def calculate_option_implied_vols(
    option_forward_panel: pd.DataFrame, lower_vol: float, upper_vol: float,
    tolerance: float, max_iterations: int, min_option_price: Optional[float] = None,
    min_iv: Optional[float] = None, max_iv: Optional[float] = None,
) -> pd.DataFrame:
    """计算全部可定价期权的IV，并为失败记录保留可审计状态。"""
    required = {
        'TRADE_DT', 'CODE', 'EXPIRY', 'EXPIRY_CODE', 'TYPE', 'STRIKE',
        'PRICE', 'VOLUME', 'OI', 'TAU', 'FORWARD', 'RISK_FREE_RATE',
    }
    missing = sorted(required - set(option_forward_panel.columns))
    if missing:
        raise ValueError(f'option_forward_panel缺少字段: {missing}')
    rows = []
    for row in option_forward_panel.itertuples(index=False):
        result = row._asdict()
        result['LOG_MONEYNESS'] = np.nan
        result['IS_OTM'] = False
        result['IV'] = np.nan
        result['IV_STATUS'] = 'INVALID_INPUT'
        try:
            if min_option_price is not None and row.PRICE < min_option_price:
                raise ValueError('LOW_PRICE')
            k = math.log(row.STRIKE / row.FORWARD)
            iv = black76_implied_volatility(
                row.PRICE, row.FORWARD, row.STRIKE, row.TAU,
                row.RISK_FREE_RATE, row.TYPE, lower_vol, upper_vol,
                tolerance, max_iterations,
            )
            if min_iv is not None and iv < min_iv:
                raise ValueError('IV_BELOW_MIN')
            if max_iv is not None and iv > max_iv:
                raise ValueError('IV_ABOVE_MAX')
            result['LOG_MONEYNESS'] = float(k)
            result['IS_OTM'] = bool(is_otm_option(row.FORWARD, row.STRIKE, row.TYPE))
            result['IV'] = float(iv)
            result['IV_STATUS'] = 'OK'
        except (TypeError, ValueError, OverflowError) as exc:
            result['IV_STATUS'] = str(exc)
        rows.append(result)
    return pd.DataFrame(rows).sort_values(
        ['TRADE_DT', 'EXPIRY', 'STRIKE', 'TYPE']
    ).reset_index(drop=True)

## 二次模型与 SVI 统一拟合接口

In [ ]:
# 拟合 sigma(k)=a+b*k+0.5*c*k^2，并令b、c直接对应一阶和二阶导数。
def fit_quadratic_model(k_values, iv_values) -> dict:
    """以最小二乘拟合二次IV曲线，必要时回退至线性或常数模型。"""
    k = np.asarray(k_values, dtype=float)
    iv = np.asarray(iv_values, dtype=float)
    if len(k) < 3:
        raise ValueError('二次模型至少需要3个样本')
    design = np.column_stack([np.ones_like(k), k, 0.5 * k * k])
    params, _, rank, _ = np.linalg.lstsq(design, iv, rcond=None)
    fitted = design @ params
    status = 'QUADRATIC'
    if rank < 3 or np.any(~np.isfinite(fitted)) or np.any(fitted <= 0):
        linear = np.column_stack([np.ones_like(k), k])
        linear_params, _, _, _ = np.linalg.lstsq(linear, iv, rcond=None)
        params = np.array([linear_params[0], linear_params[1], 0.0])
        fitted = design @ params
        status = 'FALLBACK_LINEAR'
    if np.any(~np.isfinite(fitted)) or np.any(fitted <= 0):
        params = np.array([float(np.mean(iv)), 0.0, 0.0])
        fitted = design @ params
        status = 'FALLBACK_CONSTANT'
    rmse = float(np.sqrt(np.mean((fitted - iv) ** 2)))
    return {
        'a': float(params[0]), 'b': float(params[1]), 'c': float(params[2]),
        'RMSE': rmse, 'FIT_STATUS': status,
    }

def _svi_total_variance(k, params) -> np.ndarray:
    """计算原始SVI参数化的总方差。"""
    a, b, rho, m, sigma = params
    x = np.asarray(k, dtype=float) - m
    return a + b * (rho * x + np.sqrt(x * x + sigma * sigma))

def _resolve_svi_bounds(bounds: dict, k: np.ndarray, w: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """用配置边界覆盖与数据尺度相关的SVI默认边界。"""
    span = max(float(np.ptp(k)), 0.1)
    defaults = {
        'a': (-max(float(np.max(w)), 0.1), max(float(np.max(w)) * 2.0, 0.2)),
        'b': (1e-8, max(float(np.max(w) - np.min(w)) / span * 5.0, 1.0)),
        'rho': (-0.999, 0.999), 'm': (float(np.min(k) - span), float(np.max(k) + span)),
        'sigma': (1e-4, max(span * 3.0, 1.0)),
    }
    low, high = [], []
    for name in ['a', 'b', 'rho', 'm', 'sigma']:
        configured = bounds.get(name, (None, None)) if bounds else (None, None)
        low.append(defaults[name][0] if configured[0] is None else configured[0])
        high.append(defaults[name][1] if configured[1] is None else configured[1])
    return np.asarray(low, dtype=float), np.asarray(high, dtype=float)

# 在无SciPy依赖下使用多起点有界模式搜索拟合原始SVI总方差。
def fit_svi_model(k_values, iv_values, tau: float, parameter_bounds: Optional[dict] = None) -> dict:
    """拟合SVI总方差并对非正曲线施加惩罚，失败时抛出明确错误。"""
    k = np.asarray(k_values, dtype=float)
    iv = np.asarray(iv_values, dtype=float)
    if len(k) < 5 or tau <= 0:
        raise ValueError('SVI至少需要5个样本且TAU必须为正')
    w = iv * iv * tau
    lower, upper = _resolve_svi_bounds(parameter_bounds or {}, k, w)
    center = np.array([max(float(np.min(w)) * 0.7, 0.0), 0.1, -0.2, 0.0, 0.15])
    starts = [
        center,
        np.array([float(np.min(w)) * 0.5, 0.2, -0.5, float(np.median(k)), 0.25]),
        np.array([float(np.min(w)) * 0.8, 0.05, 0.2, float(np.mean(k)), 0.10]),
    ]
    def objective(params):
        """返回SVI样本总方差的均方误差及非正值惩罚。"""
        fitted_w = _svi_total_variance(k, params)
        if np.any(~np.isfinite(fitted_w)):
            return float('inf')
        penalty = np.sum(np.minimum(fitted_w, 0.0) ** 2) * 1e8
        return float(np.mean((fitted_w - w) ** 2) + penalty)
    best_params, best_obj = None, float('inf')
    for start in starts:
        params = np.clip(start, lower, upper)
        steps = np.maximum((upper - lower) * 0.15, 1e-5)
        current = objective(params)
        for _ in range(350):
            improved = False
            for index in range(5):
                for direction in (-1.0, 1.0):
                    candidate = params.copy()
                    candidate[index] = np.clip(
                        candidate[index] + direction * steps[index], lower[index], upper[index]
                    )
                    value = objective(candidate)
                    if value < current:
                        params, current, improved = candidate, value, True
            if not improved:
                steps *= 0.5
            if float(np.max(steps)) < 1e-8:
                break
        if current < best_obj:
            best_params, best_obj = params.copy(), current
    fitted_w = _svi_total_variance(k, best_params)
    if best_params is None or np.any(fitted_w <= 0):
        raise ValueError('SVI拟合失败或产生非正总方差')
    fitted_iv = np.sqrt(fitted_w / tau)
    rmse = float(np.sqrt(np.mean((fitted_iv - iv) ** 2)))
    return {
        'a': float(best_params[0]), 'b': float(best_params[1]),
        'rho': float(best_params[2]), 'm': float(best_params[3]),
        'sigma': float(best_params[4]), 'RMSE': rmse, 'FIT_STATUS': 'SVI',
    }

# 按每日、每到期日拟合配置指定的波动率模型。
def fit_daily_volatility_models(
    option_iv_panel: pd.DataFrame, model: str, min_samples: int,
    svi_parameter_bounds: Optional[dict] = None,
) -> pd.DataFrame:
    """使用有效OTM IV逐截面拟合模型，并保留样本数、RMSE和失败原因。"""
    model = str(model).upper()
    if model not in {'QUADRATIC', 'SVI'}:
        raise ValueError('VOL_MODEL必须为QUADRATIC或SVI')
    valid = option_iv_panel.loc[
        option_iv_panel['IV_STATUS'].eq('OK')
        & option_iv_panel['IS_OTM']
        & option_iv_panel['IV'].notna()
    ].copy()
    rows = []
    for (trade_date, expiry), group in valid.groupby(['TRADE_DT', 'EXPIRY'], sort=True):
        first = group.iloc[0]
        base = {
            'TRADE_DT': trade_date, 'EXPIRY': expiry,
            'EXPIRY_CODE': first['EXPIRY_CODE'], 'TAU': float(first['TAU']),
            'FORWARD': float(first['FORWARD']),
            'RISK_FREE_RATE': float(first['RISK_FREE_RATE']),
            'MODEL': model, 'N_SAMPLES': int(len(group)),
        }
        if len(group) < min_samples:
            rows.append({**base, 'FIT_STATUS': 'INSUFFICIENT_SAMPLES', 'RMSE': np.nan})
            warnings.warn(
                f'{trade_date.date()} {first["EXPIRY_CODE"]} OTM样本不足: {len(group)}',
                RuntimeWarning,
            )
            continue
        try:
            if model == 'QUADRATIC':
                fitted = fit_quadratic_model(group['LOG_MONEYNESS'], group['IV'])
            else:
                fitted = fit_svi_model(
                    group['LOG_MONEYNESS'], group['IV'], float(first['TAU']),
                    svi_parameter_bounds,
                )
            rows.append({**base, **fitted})
        except (TypeError, ValueError, FloatingPointError) as exc:
            rows.append({**base, 'FIT_STATUS': f'FAILED: {exc}', 'RMSE': np.nan})
            warnings.warn(f'{trade_date.date()} {first["EXPIRY_CODE"]} 拟合失败: {exc}', RuntimeWarning)
    return pd.DataFrame(rows).sort_values(['TRADE_DT', 'EXPIRY']).reset_index(drop=True)

# 根据单行模型参数计算任意log-moneyness处的模型IV。
def evaluate_volatility_model(model_row, k_values) -> np.ndarray:
    """统一计算二次或SVI模型IV，并将非正或非有限结果返回为NaN。"""
    k = np.asarray(k_values, dtype=float)
    model = str(model_row['MODEL']).upper()
    if model == 'QUADRATIC':
        iv = model_row['a'] + model_row['b'] * k + 0.5 * model_row['c'] * k * k
    elif model == 'SVI':
        params = [model_row[name] for name in ['a', 'b', 'rho', 'm', 'sigma']]
        w = _svi_total_variance(k, params)
        iv = np.where(w > 0, np.sqrt(w / model_row['TAU']), np.nan)
    else:
        raise ValueError(f'未知模型: {model}')
    return np.where(np.isfinite(iv) & (iv > 0), iv, np.nan)

## Forward-Delta 的25C、75P与ATM定位

In [ ]:
def _bisect_root(function: Callable[[float], float], low: float, high: float, tolerance: float = 1e-10) -> float:
    """在已知异号区间内使用二分法求标量函数根。"""
    f_low, f_high = function(low), function(high)
    if not np.isfinite(f_low) or not np.isfinite(f_high) or f_low * f_high > 0:
        raise ValueError('根搜索区间未形成有效异号包围')
    for _ in range(200):
        mid = 0.5 * (low + high)
        f_mid = function(mid)
        if abs(f_mid) <= tolerance or high - low <= tolerance:
            return float(mid)
        if f_low * f_mid <= 0:
            high, f_high = mid, f_mid
        else:
            low, f_low = mid, f_mid
    return float(0.5 * (low + high))

# 使用拟合曲面和Black-76 Forward Delta寻找指定目标Delta对应的连续执行价。
def find_forward_delta_landmark(model_row, target_delta: float, option_type: str) -> dict:
    """在log-moneyness空间求解Forward Delta目标并返回k、Strike、IV和误差。"""
    F = float(model_row['FORWARD'])
    tau = float(model_row['TAU'])
    r = float(model_row['RISK_FREE_RATE'])
    option_type = _normalize_option_type(option_type)
    side_low, side_high = (0.0, 2.0) if option_type == 'CALL' else (-2.0, 0.0)
    grid = np.linspace(side_low, side_high, 1001)
    def objective(k):
        """计算给定k处的Forward Delta与目标Delta之差。"""
        iv = float(evaluate_volatility_model(model_row, [k])[0])
        if not np.isfinite(iv):
            return np.nan
        strike = F * math.exp(k)
        return black76_delta(F, strike, iv, tau, r, option_type) - target_delta
    values = np.array([objective(k) for k in grid])
    brackets = []
    for index in range(len(grid) - 1):
        if np.isfinite(values[index]) and np.isfinite(values[index + 1]):
            if values[index] == 0 or values[index] * values[index + 1] < 0:
                brackets.append((grid[index], grid[index + 1]))
    if not brackets:
        finite = np.isfinite(values)
        if not finite.any():
            raise ValueError(f'找不到Forward Delta={target_delta}的有效搜索点')
        best_index = int(np.nanargmin(np.where(finite, np.abs(values), np.nan)))
        k_star = float(grid[best_index])
        status = 'FALLBACK_NEAREST'
    else:
        low, high = min(brackets, key=lambda pair: abs(0.5 * (pair[0] + pair[1])))
        k_star = _bisect_root(objective, low, high)
        status = 'OK'
    iv_star = float(evaluate_volatility_model(model_row, [k_star])[0])
    strike = F * math.exp(k_star)
    achieved = black76_delta(F, strike, iv_star, tau, r, option_type)
    return {
        'TARGET_DELTA': float(target_delta), 'LOG_MONEYNESS': k_star,
        'STRIKE': float(strike), 'MODEL_IV': iv_star,
        'ACHIEVED_DELTA': float(achieved),
        'DELTA_ERROR': float(abs(achieved - target_delta)), 'STATUS': status,
    }

# 为每个成功拟合截面生成90P、75P、ATM、25C和10C五个IV曲线定位点。
def build_delta_landmarks(model_parameters: pd.DataFrame, target_deltas: dict) -> pd.DataFrame:
    """使用Forward Delta定位翼部，并以k=0构造ATM基准点。"""
    rows = []
    for _, model_row in model_parameters.iterrows():
        if str(model_row['FIT_STATUS']).startswith(('FAILED', 'INSUFFICIENT')):
            continue
        base = {
            'TRADE_DT': model_row['TRADE_DT'], 'EXPIRY': model_row['EXPIRY'],
            'EXPIRY_CODE': model_row['EXPIRY_CODE'], 'MODEL': model_row['MODEL'],
        }
        atm_iv = float(evaluate_volatility_model(model_row, [ATM_LOG_MONEYNESS])[0])
        rows.append({
            **base, 'TARGET': 'ATM', 'TARGET_DELTA': np.nan,
            'LOG_MONEYNESS': float(ATM_LOG_MONEYNESS),
            'STRIKE': float(model_row['FORWARD']), 'MODEL_IV': atm_iv,
            'ACHIEVED_DELTA': np.nan, 'DELTA_ERROR': np.nan, 'STATUS': 'OK',
        })
        for target, option_type in [
            ('90PUT', 'PUT'), ('75PUT', 'PUT'),
            ('25CALL', 'CALL'), ('10CALL', 'CALL'),
        ]:
            try:
                landmark = find_forward_delta_landmark(
                    model_row, target_deltas[target], option_type
                )
            except (TypeError, ValueError, FloatingPointError) as exc:
                landmark = {
                    'TARGET_DELTA': target_deltas[target], 'LOG_MONEYNESS': np.nan,
                    'STRIKE': np.nan, 'MODEL_IV': np.nan, 'ACHIEVED_DELTA': np.nan,
                    'DELTA_ERROR': np.nan, 'STATUS': f'FAILED: {exc}',
                }
                warnings.warn(
                    f'{model_row["TRADE_DT"].date()} {model_row["EXPIRY_CODE"]} {target}定位失败: {exc}',
                    RuntimeWarning,
                )
            rows.append({**base, 'TARGET': target, **landmark})
    return pd.DataFrame(rows).sort_values(['TRADE_DT', 'EXPIRY', 'TARGET']).reset_index(drop=True)

# 根据五个Forward-Delta定位点计算25D Skew、25D Butterfly和10D Skew。
def build_smile_metrics(delta_landmarks: pd.DataFrame) -> pd.DataFrame:
    """将定位点透视为每个截面的标准翼部波动率指标表。"""
    valid = delta_landmarks.loc[~delta_landmarks['STATUS'].astype(str).str.startswith('FAILED')].copy()
    pivot = valid.pivot_table(
        index=['TRADE_DT', 'EXPIRY', 'EXPIRY_CODE', 'MODEL'],
        columns='TARGET', values='MODEL_IV', aggfunc='first',
    ).reset_index()
    required = ['90PUT', '75PUT', 'ATM', '25CALL', '10CALL']
    for column in required:
        if column not in pivot:
            pivot[column] = np.nan
    pivot['SKEW_25D'] = pivot['25CALL'] - pivot['75PUT']
    pivot['BF_25D'] = 0.5 * (pivot['25CALL'] + pivot['75PUT']) - pivot['ATM']
    pivot['SKEW_10D'] = pivot['10CALL'] - pivot['90PUT']
    return pivot.sort_values(['TRADE_DT', 'EXPIRY']).reset_index(drop=True)

In [ ]:
# 按配置口径从04已有拟合结果中提取每个交易日、每个到期日的Skew。
def build_skew_panel(
    model_parameters: pd.DataFrame, smile_metrics: pd.DataFrame, skew_method: str,
) -> pd.DataFrame:
    """统一生成供后续模块直接使用的TRADE_DT、EXPIRY、SKEW面板。"""
    method = str(skew_method).upper()
    if method == 'DELTA':
        panel = smile_metrics[['TRADE_DT', 'EXPIRY', 'EXPIRY_CODE', 'MODEL', 'SKEW_25D']].copy()
        panel = panel.rename(columns={'SKEW_25D': 'SKEW'})
    elif method == 'DERIVATIVE':
        panel = model_parameters.copy()
        for column in ['a', 'b', 'rho', 'm', 'sigma']:
            if column not in panel.columns:
                panel[column] = np.nan
        panel = panel[['TRADE_DT', 'EXPIRY', 'EXPIRY_CODE', 'MODEL', 'TAU', 'a', 'b', 'rho', 'm', 'sigma']]
        panel['SKEW'] = np.nan
        quadratic = panel['MODEL'].eq('QUADRATIC')
        panel.loc[quadratic, 'SKEW'] = panel.loc[quadratic, 'b']
        svi = panel['MODEL'].eq('SVI')
        if svi.any():
            svi_rows = panel.loc[svi]
            root = np.sqrt(svi_rows['m'] ** 2 + svi_rows['sigma'] ** 2)
            w0 = svi_rows['a'] + svi_rows['b'] * (-svi_rows['rho'] * svi_rows['m'] + root)
            iv0 = np.sqrt(w0 / svi_rows['TAU'])
            dw_dk = svi_rows['b'] * (svi_rows['rho'] - svi_rows['m'] / root)
            panel.loc[svi, 'SKEW'] = dw_dk / (2.0 * svi_rows['TAU'] * iv0)
        panel = panel[['TRADE_DT', 'EXPIRY', 'EXPIRY_CODE', 'MODEL', 'SKEW']]
    else:
        raise ValueError("SKEW_METHOD must be 'DELTA' or 'DERIVATIVE'")
    panel['SKEW_METHOD'] = method
    return panel.sort_values(['TRADE_DT', 'EXPIRY']).reset_index(drop=True)


## 每日多到期日 IV 曲线总图

每个交易日保存一张总图，当天所有到期月份按两列分面展示。横轴为执行价 $K$，竖线依次标记90P、75P、ATM、25C和10C；标题区展示模型参数、25D Skew、25D BF和10D Skew。

In [ ]:
# 将同一交易日的所有到期月份组合成一张多分面IV曲线总图。
def plot_iv_curves(
    option_iv_panel: pd.DataFrame, model_parameters: pd.DataFrame,
    delta_landmarks: pd.DataFrame, smile_metrics: pd.DataFrame,
    figure_path, figure_format: str = 'png'
) -> list[Path]:
    """每天生成一张包含全部期限分面的IV曲线图。"""
    if plt is None:
        warnings.warn('matplotlib不可用，未生成IV曲线图片', RuntimeWarning)
        return []
    figure_path = Path(figure_path)
    figure_path.mkdir(parents=True, exist_ok=True)
    saved = []
    valid_iv = option_iv_panel.loc[
        option_iv_panel['IV_STATUS'].eq('OK') & option_iv_panel['IS_OTM']
    ]
    successful = model_parameters.loc[
        ~model_parameters['FIT_STATUS'].astype(str).str.startswith(('FAILED', 'INSUFFICIENT'))
    ]
    for trade_date, daily_models in successful.groupby('TRADE_DT', sort=True):
        daily_models = daily_models.sort_values('EXPIRY').reset_index(drop=True)
        n_panels = len(daily_models)
        n_cols, n_rows = 2, int(math.ceil(n_panels / 2))
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, 4.6 * n_rows + 1.0), squeeze=False)
        fig.suptitle(
            f'MO OTM IV Smile - {trade_date:%Y%m%d} | {MODEL_SKEW_TAG}',
            fontsize=16, fontweight='bold', x=0.06, ha='left',
        )
        for panel_index, (_, model_row) in enumerate(daily_models.iterrows()):
            ax = axes.flat[panel_index]
            group = valid_iv.loc[
                valid_iv['TRADE_DT'].eq(trade_date)
                & valid_iv['EXPIRY'].eq(model_row['EXPIRY'])
            ]
            marks = delta_landmarks.loc[
                delta_landmarks['TRADE_DT'].eq(trade_date)
                & delta_landmarks['EXPIRY'].eq(model_row['EXPIRY'])
                & ~delta_landmarks['STATUS'].astype(str).str.startswith('FAILED')
            ]
            metrics = smile_metrics.loc[
                smile_metrics['TRADE_DT'].eq(trade_date)
                & smile_metrics['EXPIRY'].eq(model_row['EXPIRY'])
            ]
            metric = metrics.iloc[0] if not metrics.empty else None
            mark_k = marks['LOG_MONEYNESS'].dropna().tolist()
            k_min = min(group['LOG_MONEYNESS'].min(), min(mark_k) if mark_k else 0.0) - 0.025
            k_max = max(group['LOG_MONEYNESS'].max(), max(mark_k) if mark_k else 0.0) + 0.025
            k_grid = np.linspace(k_min, k_max, 400)
            fitted_iv = evaluate_volatility_model(model_row, k_grid)
            strike_grid = float(model_row['FORWARD']) * np.exp(k_grid)
            for option_type, color in [('PUT', '#ef5350'), ('CALL', '#3f7de8')]:
                sample = group.loc[group['TYPE'].eq(option_type)]
                ax.scatter(sample['STRIKE'], sample['IV'], s=20, color=color, alpha=0.95)
            ax.plot(strike_grid, fitted_iv, color='#172033', linewidth=1.8)
            label_map = {
                '90PUT': '90P', '75PUT': '75P', 'ATM': 'ATM',
                '25CALL': '25C', '10CALL': '10C',
            }
            for _, mark in marks.iterrows():
                is_atm = mark['TARGET'] == 'ATM'
                ax.axvline(
                    mark['STRIKE'], color='#55789e',
                    linestyle='--', linewidth=1.25 if is_atm else 0.8, alpha=0.9,
                )
                ax.text(
                    mark['STRIKE'], 0.985, label_map[mark['TARGET']],
                    transform=ax.get_xaxis_transform(), ha='center', va='top', fontsize=7,
                )
            if model_row['MODEL'] == 'QUADRATIC':
                param_text = (
                    f"a={model_row['a']:.2%} b={model_row['b']:.4f} c={model_row['c']:.4f}"
                )
                formula_text = 'sigma(k)=a+b*k+0.5*c*k^2'
            else:
                param_text = (
                    f"a={model_row['a']:.4f} b={model_row['b']:.4f} rho={model_row['rho']:.3f} "
                    f"m={model_row['m']:.4f} sigma={model_row['sigma']:.4f}"
                )
                formula_text = 'SVI total variance'
            metric_text = (
                f"25D skew={metric['SKEW_25D']:.2%} | BF={metric['BF_25D']:.2%} | "
                f"10D skew={metric['SKEW_10D']:.2%}"
                if metric is not None else 'Smile metrics unavailable'
            )
            ax.set_title(
                f"MO{model_row['EXPIRY_CODE']} exp {model_row['EXPIRY']:%Y-%m-%d} | {param_text}\n"
                f"{metric_text}\n{formula_text} | N={int(model_row['N_SAMPLES'])} | RMSE={model_row['RMSE']:.2%}",
                loc='left', fontsize=8.5, pad=10,
            )
            ax.set_xlabel('Strike K', fontsize=8)
            ax.set_ylabel('IV', fontsize=8)
            ax.yaxis.set_major_formatter(lambda value, position: f'{value:.1%}')
            ax.grid(color='#d9e0e8', linewidth=0.7, alpha=0.8)
            ax.tick_params(labelsize=7)
        for unused in range(n_panels, n_rows * n_cols):
            axes.flat[unused].axis('off')
        fig.tight_layout(rect=[0, 0, 1, 0.97], h_pad=2.5, w_pad=1.8)
        output = figure_path / f'{trade_date:%Y%m%d}_all_expiries.{figure_format}'
        fig.savefig(output, dpi=160, bbox_inches='tight', facecolor='white')
        plt.close(fig)
        saved.append(output)
    print(f'IV curve figures saved: {len(saved)}')
    return saved

## 质量检查、保存与统一入口

In [ ]:
# 检查IV成功率、模型参数、样本数和Forward-Delta定位误差。
def run_volatility_model_quality_checks(
    option_iv_panel: pd.DataFrame, model_parameters: pd.DataFrame,
    delta_landmarks: pd.DataFrame, smile_metrics: pd.DataFrame, min_samples: int,
) -> pd.DataFrame:
    """生成逐项质量检查表，并对失败检查给出明确warning。"""
    successful_models = ~model_parameters['FIT_STATUS'].astype(str).str.startswith(
        ('FAILED', 'INSUFFICIENT')
    )
    wing_marks = delta_landmarks.loc[
        delta_landmarks['TARGET'].isin(['90PUT', '75PUT', '25CALL', '10CALL'])
    ]
    successful_marks = ~wing_marks['STATUS'].astype(str).str.startswith('FAILED')
    exact_marks = wing_marks['STATUS'].eq('OK')
    max_delta_error = wing_marks.loc[exact_marks, 'DELTA_ERROR'].max()
    checks = [
        ('存在有效IV', option_iv_panel['IV_STATUS'].eq('OK').any(), f"ok={int(option_iv_panel['IV_STATUS'].eq('OK').sum())}/{len(option_iv_panel)}"),
        ('存在OTM拟合样本', (option_iv_panel['IV_STATUS'].eq('OK') & option_iv_panel['IS_OTM']).any(), 'OTM close-price IV'),
        ('存在成功模型', successful_models.any(), f"success={int(successful_models.sum())}/{len(model_parameters)}"),
        ('成功模型样本充分', model_parameters.loc[successful_models, 'N_SAMPLES'].ge(min_samples).all(), f'min={min_samples}'),
        ('成功模型RMSE有限', np.isfinite(model_parameters.loc[successful_models, 'RMSE']).all(), '无NaN/Inf'),
        ('ATM全部成功', delta_landmarks.loc[delta_landmarks['TARGET'].eq('ATM'), 'STATUS'].eq('OK').all(), 'k=0'),
        ('Forward Delta定位可用', successful_marks.all(), f"available={int(successful_marks.sum())}/{len(wing_marks)}"),
        ('精确Forward Delta误差', pd.notna(max_delta_error) and max_delta_error <= 1e-7, f"max={max_delta_error:.3e}; fallback={int((wing_marks['STATUS'] == 'FALLBACK_NEAREST').sum())}"),
        ('Smile指标完整', smile_metrics[['SKEW_25D', 'BF_25D', 'SKEW_10D']].notna().all().all(), f'rows={len(smile_metrics)}'),
    ]
    table = pd.DataFrame(checks, columns=['CHECK', 'PASSED', 'DETAIL'])
    failed = table.loc[~table['PASSED']]
    if not failed.empty:
        warnings.warn(f'波动率模型质量检查失败: {failed["CHECK"].tolist()}', RuntimeWarning)
    display(table)
    return table

# 保存期权IV、模型参数、Forward-Delta定位点、Smile指标和标准Skew面板。
def save_volatility_model_results(
    option_iv_panel: pd.DataFrame, model_parameters: pd.DataFrame,
    delta_landmarks: pd.DataFrame, smile_metrics: pd.DataFrame,
    skew_panel: pd.DataFrame, output_path,
) -> dict:
    """以UTF-8 CSV保存04模块的五个核心结果表。"""
    output_path = Path(output_path)
    output_path.mkdir(parents=True, exist_ok=True)
    files = {
        'option_iv_panel': output_path / 'option_iv_panel.csv',
        'model_parameters': output_path / 'volatility_model_parameters.csv',
        'delta_landmarks': output_path / 'delta_landmarks.csv',
        'smile_metrics': output_path / 'smile_metrics.csv',
        'skew_panel': output_path / 'skew_panel.csv',
    }
    for name, frame in {
        'option_iv_panel': option_iv_panel, 'model_parameters': model_parameters,
        'delta_landmarks': delta_landmarks, 'smile_metrics': smile_metrics,
        'skew_panel': skew_panel,
    }.items():
        frame.to_csv(files[name], index=False, encoding='utf-8-sig', date_format='%Y-%m-%d')
    print('Volatility model results saved:')
    for name, path in files.items():
        print(f'  {name}: {path}')
    return files

# 按配置完成IV反解、逐截面拟合和Forward-Delta定位。
def build_volatility_model_data(verbose: bool = True):
    """生成并返回IV面板、模型参数、Delta定位点、Smile指标和Skew面板。"""
    if VOL_SURFACE_DELTA_BASIS != 'FORWARD':
        raise ValueError("04模块要求VOL_SURFACE_DELTA_BASIS='FORWARD'")
    iv_panel = calculate_option_implied_vols(
        option_forward_panel, IV_SOLVER_LOWER_BOUND, MAX_IV,
        IV_SOLVER_TOLERANCE, IV_SOLVER_MAX_ITERATIONS,
        MIN_OPTION_PRICE, MIN_IV, MAX_IV,
    )
    parameters = fit_daily_volatility_models(
        iv_panel, VOL_MODEL, MIN_OTM_OPTIONS_PER_EXPIRY, SVI_PARAMETER_BOUNDS
    )
    landmarks = build_delta_landmarks(parameters, VOL_SURFACE_TARGET_DELTAS)
    metrics = build_smile_metrics(landmarks)
    skew = build_skew_panel(parameters, metrics, SKEW_METHOD)
    if verbose:
        print('IV calculation completed')
        print(iv_panel['IV_STATUS'].value_counts().head(10))
        display(iv_panel.head())
        print('Volatility model calibration completed')
        display(parameters.head())
        print('Forward-Delta landmarks completed')
        display(landmarks.head(15))
        print('Smile metrics completed')
        display(metrics.head())
        print(f'Skew panel completed: {SKEW_METHOD}')
        display(skew.head())
    return iv_panel, parameters, landmarks, metrics, skew

## 执行与结果

In [ ]:
option_iv_panel, volatility_model_parameters, delta_landmarks, smile_metrics, skew_panel = (
    build_volatility_model_data(verbose=True)
)
volatility_model_quality_checks = run_volatility_model_quality_checks(
    option_iv_panel, volatility_model_parameters, delta_landmarks, smile_metrics,
    MIN_OTM_OPTIONS_PER_EXPIRY,
)
iv_curve_files = (
    plot_iv_curves(
        option_iv_panel, volatility_model_parameters, delta_landmarks, smile_metrics,
        IV_CURVE_FIGURE_PATH, FIGURE_FORMAT,
    ) if SAVE_FIGURE else []
)
volatility_model_files = (
    save_volatility_model_results(
        option_iv_panel, volatility_model_parameters, delta_landmarks, smile_metrics,
        skew_panel,
        VOLATILITY_MODEL_OUTPUT_PATH,
    ) if SAVE_CSV else {}
)